In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
filename = "train_service_passenger_counts_fy_2024_2025.csv"

print(os.path.exists(filename))

True


In [3]:
file_size_gb = os.path.getsize(filename) / (1024**3)

print(f"File size: {file_size_gb:.2f} GB")

File size: 2.64 GB


In [4]:
sample = pd.read_csv(filename, nrows=1000)

print("Rows and columns:", sample.shape)

Rows and columns: (1000, 21)


In [5]:
print(sample.columns.tolist())

['Business_Date', 'Day_of_Week', 'Day_Type', 'Mode', 'Train_Number', 'Line_Name', 'Group', 'Direction', 'Origin_Station', 'Destination_Station', 'Station_Name', 'Station_Latitude', 'Station_Longitude', 'Station_Chainage', 'Stop_Sequence_Number', 'Arrival_Time_Scheduled', 'Departure_Time_Scheduled', 'Passenger_Boardings', 'Passenger_Alightings', 'Passenger_Arrival_Load', 'Passenger_Departure_Load']


In [6]:
sample.head(10)

,Business_Date,Day_of_Week,Day_Type,Mode,Train_Number,Line_Name,Group,Direction,Origin_Station,Destination_Station,...,Station_Latitude,Station_Longitude,Station_Chainage,Stop_Sequence_Number,Arrival_Time_Scheduled,Departure_Time_Scheduled,Passenger_Boardings,Passenger_Alightings,Passenger_Arrival_Load,Passenger_Departure_Load
0,2024-07-01,Monday,School Holiday,Metro Trains,2311,Alamein,Alamein-Riversdale,D,Camberwell,Alamein,...,-37.851563,145.080511,14170,5,23:56:40,23:57:00,0,0,10,0
1,2024-07-01,Monday,School Holiday,Metro Trains,2311,Alamein,Alamein-Riversdale,D,Camberwell,Alamein,...,-37.843985,145.075560,13303,4,23:55:40,23:56:00,0,0,10,10
2,2024-07-01,Monday,School Holiday,Metro Trains,2311,Alamein,Alamein-Riversdale,D,Camberwell,Alamein,...,-37.861968,145.081344,15392,6,23:58:40,23:59:00,0,0,0,0
3,2024-07-01,Monday,School Holiday,Metro Trains,2311,Alamein,Alamein-Riversdale,D,Camberwell,Alamein,...,-37.831505,145.069646,11733,2,23:52:40,23:53:00,0,0,10,10
4,2024-07-01,Monday,School Holiday,Metro Trains,2311,Alamein,Alamein-Riversdale,D,Camberwell,Alamein,...,-37.835716,145.070298,12151,3,23:53:40,23:54:00,0,0,10,10
5,2024-07-01,Monday,School Holiday,Metro Trains,2311,Alamein,Alamein-Riversdale,D,Camberwell,Alamein,...,-37.826567,145.058697,10205,1,23:50:00,23:50:00,10,0,0,10
6,2024-07-01,Monday,School Holiday,Metro Trains,2311,Alamein,Alamein-Riversdale,D,Camberwell,Alamein,...,-37.868320,145.079656,16121,7,00:01:00,00:01:00,0,0,0,0
7,2024-07-01,Monday,School Holiday,Metro Trains,2339,Alamein,Alamein-Riversdale,D,Camberwell,Alamein,...,-37.843985,145.075560,13303,4,03:27:40,03:28:00,0,0,10,10
8,2024-07-01,Monday,School Holiday,Metro Trains,2339,Alamein,Alamein-Riversdale,D,Camberwell,Alamein,...,-37.826567,145.058697,10205,1,03:22:00,03:22:00,10,0,0,10
9,2024-07-01,Monday,School Holiday,Metro Trains,2339,Alamein,Alamein-Riversdale,D,Camberwell,Alamein,...,-37.861968,145.081344,15392,6,03:30:40,03:31:00,0,10,10,0


In [7]:
sample.dtypes

Business_Date                object
Day_of_Week                  object
Day_Type                     object
Mode                         object
Train_Number                 object
Line_Name                    object
Group                        object
Direction                    object
Origin_Station               object
Destination_Station          object
Station_Name                 object
Station_Latitude            float64
Station_Longitude           float64
Station_Chainage              int64
Stop_Sequence_Number          int64
Arrival_Time_Scheduled       object
Departure_Time_Scheduled     object
Passenger_Boardings           int64
Passenger_Alightings          int64
Passenger_Arrival_Load        int64
Passenger_Departure_Load      int64
dtype: object

In [8]:
sample['Business_Date'] = pd.to_datetime(
    sample['Business_Date'],
    errors='coerce'
)

print(sample['Business_Date'].dtype)

datetime64[ns]


In [9]:
print("Earliest date in sample:", sample['Business_Date'].min())
print("Latest date in sample:", sample['Business_Date'].max())

Earliest date in sample: 2024-07-01 00:00:00
Latest date in sample: 2024-07-02 00:00:00


In [10]:
date_min = None
date_max = None

for chunk in pd.read_csv(
    filename,
    usecols=['Business_Date'],
    chunksize=100000
):
    dates = pd.to_datetime(
        chunk['Business_Date'],
        errors='coerce'
    )

    chunk_min = dates.min()
    chunk_max = dates.max()

    if date_min is None or chunk_min < date_min:
        date_min = chunk_min

    if date_max is None or chunk_max > date_max:
        date_max = chunk_max

print("Full dataset earliest date:", date_min)
print("Full dataset latest date:", date_max)

Full dataset earliest date: 2024-07-01 00:00:00
Full dataset latest date: 2025-06-30 00:00:00


In [11]:
stations = set()

for chunk in pd.read_csv(
    filename,
    usecols=['Station_Name'],
    chunksize=100000
):
    stations.update(
        chunk['Station_Name'].dropna().unique()
    )

print("Number of unique stations:", len(stations))

Number of unique stations: 315


In [12]:
train_services = set()

for chunk in pd.read_csv(
    filename,
    usecols=['Train_Number'],
    chunksize=100000
):
    train_services.update(
        chunk['Train_Number'].dropna().unique()
    )

print("Number of unique train service identifiers:", len(train_services))

Number of unique train service identifiers: 5878


In [13]:
lines = set()

for chunk in pd.read_csv(
    filename,
    usecols=['Line_Name'],
    chunksize=100000
):
    lines.update(
        chunk['Line_Name'].dropna().unique()
    )

print("Number of unique train lines:", len(lines))

Number of unique train lines: 23


In [64]:
coordinate_missing = {
    'Station_Latitude': 0,
    'Station_Longitude': 0
}

for chunk in pd.read_csv(
    filename,
    usecols=[
        'Station_Latitude',
        'Station_Longitude'
    ],
    chunksize=100000
):
    coordinate_missing['Station_Latitude'] += (
        chunk['Station_Latitude'].isna().sum()
    )

    coordinate_missing['Station_Longitude'] += (
        chunk['Station_Longitude'].isna().sum()
    )

print(pd.Series(coordinate_missing))

Station_Latitude     0
Station_Longitude    0
dtype: int64


In [67]:
missing_counts = {}
total_rows = 0

for chunk in pd.read_csv(
    filename,
    chunksize=100000
):
    total_rows += len(chunk)

    for col in chunk.columns:
        if col not in missing_counts:
            missing_counts[col] = 0

        missing_counts[col] += chunk[col].isna().sum()

missing_summary = pd.DataFrame({
    'Missing_Count': missing_counts
})

missing_summary['Missing_Percentage'] = (
    missing_summary['Missing_Count']
    / total_rows * 100
).round(2)

missing_summary

,Missing_Count,Missing_Percentage
Business_Date,0,0.00
Day_of_Week,0,0.00
Day_Type,0,0.00
Mode,0,0.00
Train_Number,0,0.00
Line_Name,0,0.00
Group,0,0.00
Direction,0,0.00
Origin_Station,0,0.00
Destination_Station,0,0.00


In [18]:
passenger_columns = [
    'Passenger_Boardings',
    'Passenger_Alightings',
    'Passenger_Arrival_Load',
    'Passenger_Departure_Load'
]

zero_counts = {col: 0 for col in passenger_columns}
total_rows = 0

for chunk in pd.read_csv(
    filename,
    usecols=passenger_columns,
    chunksize=100000
):
    total_rows += len(chunk)

    for col in passenger_columns:
        zero_counts[col] += (chunk[col] == 0).sum()

zero_summary = pd.DataFrame({
    'Zero_Count': zero_counts
})

zero_summary['Zero_Percentage'] = (
    zero_summary['Zero_Count'] / total_rows * 100
).round(2)

zero_summary

,Zero_Count,Zero_Percentage
Passenger_Boardings,8203267,51.47
Passenger_Alightings,8150176,51.14
Passenger_Arrival_Load,1598244,10.03
Passenger_Departure_Load,1598312,10.03


In [62]:
duplicate_columns = [
    'Business_Date',
    'Train_Number',
    'Station_Name',
    'Stop_Sequence_Number'
]

seen = set()
duplicate_count = 0
total_rows = 0

for chunk in pd.read_csv(
    filename,
    usecols=duplicate_columns,
    chunksize=100000
):
    keys = list(
        zip(
            chunk['Business_Date'],
            chunk['Train_Number'],
            chunk['Station_Name'],
            chunk['Stop_Sequence_Number']
        )
    )

    for key in keys:
        if key in seen:
            duplicate_count += 1
        else:
            seen.add(key)

    total_rows += len(chunk)

print("Total rows:", total_rows)
print("Duplicate records found:", duplicate_count)

Total rows: 15937506
Duplicate records found: 0


In [20]:
# Examine one train service on one business date
example = pd.read_csv(
    filename,
    usecols=[
        'Business_Date',
        'Train_Number',
        'Line_Name',
        'Origin_Station',
        'Destination_Station',
        'Station_Name',
        'Stop_Sequence_Number',
        'Passenger_Boardings',
        'Passenger_Alightings'
    ],
    nrows=1000
)

example.head(20)

,Business_Date,Train_Number,Line_Name,Origin_Station,Destination_Station,Station_Name,Stop_Sequence_Number,Passenger_Boardings,Passenger_Alightings
0,2024-07-01,2311,Alamein,Camberwell,Alamein,Burwood,5,0,0
1,2024-07-01,2311,Alamein,Camberwell,Alamein,Hartwell,4,0,0
2,2024-07-01,2311,Alamein,Camberwell,Alamein,Ashburton,6,0,0
3,2024-07-01,2311,Alamein,Camberwell,Alamein,Riversdale,2,0,0
4,2024-07-01,2311,Alamein,Camberwell,Alamein,Willison,3,0,0
5,2024-07-01,2311,Alamein,Camberwell,Alamein,Camberwell,1,10,0
6,2024-07-01,2311,Alamein,Camberwell,Alamein,Alamein,7,0,0
7,2024-07-01,2339,Alamein,Camberwell,Alamein,Hartwell,4,0,0
8,2024-07-01,2339,Alamein,Camberwell,Alamein,Camberwell,1,10,0
9,2024-07-01,2339,Alamein,Camberwell,Alamein,Ashburton,6,0,10


In [66]:
passenger_columns = [
    'Passenger_Boardings',
    'Passenger_Alightings',
    'Passenger_Arrival_Load',
    'Passenger_Departure_Load'
]

passenger_stats = []

for chunk in pd.read_csv(
    filename,
    usecols=passenger_columns,
    chunksize=100000
):
    passenger_stats.append(chunk)

passenger_data = pd.concat(
    passenger_stats,
    ignore_index=True
)

passenger_data.describe()

,Passenger_Boardings,Passenger_Alightings,Passenger_Arrival_Load,Passenger_Departure_Load
count,1.593751e+07,1.593751e+07,1.593751e+07,1.593751e+07
mean,1.557092e+01,1.556413e+01,1.152071e+02,1.152083e+02
std,3.598534e+01,3.714762e+01,1.268846e+02,1.268843e+02
min,0.000000e+00,-9.000000e+01,-9.000000e+01,-9.000000e+01
25%,0.000000e+00,0.000000e+00,3.000000e+01,3.000000e+01
50%,0.000000e+00,0.000000e+00,8.000000e+01,8.000000e+01
75%,2.000000e+01,2.000000e+01,1.600000e+02,1.600000e+02
max,2.370000e+03,2.570000e+03,2.630000e+03,2.630000e+03


In [22]:
negative_counts = {
    'Passenger_Boardings': 0,
    'Passenger_Alightings': 0,
    'Passenger_Arrival_Load': 0,
    'Passenger_Departure_Load': 0
}

for chunk in pd.read_csv(
    filename,
    usecols=[
        'Passenger_Boardings',
        'Passenger_Alightings',
        'Passenger_Arrival_Load',
        'Passenger_Departure_Load'
    ],
    chunksize=100000
):
    for col in negative_counts:
        negative_counts[col] += (chunk[col] < 0).sum()

print(negative_counts)

{'Passenger_Boardings': np.int64(0), 'Passenger_Alightings': np.int64(2), 'Passenger_Arrival_Load': np.int64(16), 'Passenger_Departure_Load': np.int64(16)}


In [23]:
negative_records = []

use_columns = [
    'Business_Date',
    'Train_Number',
    'Line_Name',
    'Station_Name',
    'Stop_Sequence_Number',
    'Passenger_Boardings',
    'Passenger_Alightings',
    'Passenger_Arrival_Load',
    'Passenger_Departure_Load'
]

for chunk in pd.read_csv(
    filename,
    usecols=use_columns,
    chunksize=100000
):
    negative_rows = chunk[
        (chunk['Passenger_Boardings'] < 0) |
        (chunk['Passenger_Alightings'] < 0) |
        (chunk['Passenger_Arrival_Load'] < 0) |
        (chunk['Passenger_Departure_Load'] < 0)
    ]

    if not negative_rows.empty:
        negative_records.append(negative_rows)

negative_df = pd.concat(
    negative_records,
    ignore_index=True
)

negative_df

,Business_Date,Train_Number,Line_Name,Station_Name,Stop_Sequence_Number,Passenger_Boardings,Passenger_Alightings,Passenger_Arrival_Load,Passenger_Departure_Load
0,2025-03-16,8610,North East,Benalla,6,30,50,-50,-70
1,2025-03-16,8610,North East,Avenel,9,0,10,-80,-90
2,2025-03-16,8610,North East,Wangaratta,5,70,100,-10,-50
3,2025-03-16,8610,North East,Euroa,8,10,30,-70,-80
4,2025-03-16,8610,North East,Springhurst,4,0,20,10,-10
5,2025-03-16,8610,North East,Southern Cross,12,0,-70,-70,0
6,2025-03-16,8610,North East,Broadmeadows,11,0,0,-70,-70
7,2025-03-16,8610,North East,Violet Town,7,0,0,-70,-70
8,2025-03-16,8610,North East,Seymour,10,20,0,-90,-70
9,2025-03-30,8610,North East,Benalla,6,30,50,-50,-70


In [24]:
negative_df.shape

(18, 9)

In [25]:
negative_df.to_string(index=False)

'Business_Date Train_Number  Line_Name   Station_Name  Stop_Sequence_Number  Passenger_Boardings  Passenger_Alightings  Passenger_Arrival_Load  Passenger_Departure_Load\n   2025-03-16         8610 North East        Benalla                     6                   30                    50                     -50                       -70\n   2025-03-16         8610 North East         Avenel                     9                    0                    10                     -80                       -90\n   2025-03-16         8610 North East     Wangaratta                     5                   70                   100                     -10                       -50\n   2025-03-16         8610 North East          Euroa                     8                   10                    30                     -70                       -80\n   2025-03-16         8610 North East    Springhurst                     4                    0                    20                      10             

In [26]:
line_boardings = {}

for chunk in pd.read_csv(
    filename,
    usecols=['Line_Name', 'Passenger_Boardings'],
    chunksize=100000
):
    grouped = chunk.groupby('Line_Name')['Passenger_Boardings'].sum()

    for line, value in grouped.items():
        line_boardings[line] = line_boardings.get(line, 0) + value

line_boardings_df = (
    pd.Series(line_boardings)
    .sort_values(ascending=False)
)

line_boardings_df

Pakenham                  24089050
Frankston                 20748810
Lilydale                  20067740
Craigieburn               18066770
Mernda                    18032940
Werribee                  17931300
Cranbourne                17072380
Sunbury                   16266180
Hurstbridge               13862590
Belgrave                  13760130
Glen Waverley             12291120
Sandringham               11932610
South Western             10936490
Upfield                    9361060
Western                    6518010
Williamstown               5786580
Alamein                    3686730
Northern                   2807910
North East                 1881930
Eastern                    1642430
Flemington Racecourse       703700
Richmond and City Loop      645810
Stony Point                  69350
dtype: int64

In [28]:
station_boardings = {}
station_alightings = {}

for chunk in pd.read_csv(
    filename,
    usecols=[
        'Station_Name',
        'Passenger_Boardings',
        'Passenger_Alightings'
    ],
    chunksize=100000
):
    boarding_totals = chunk.groupby(
        'Station_Name'
    )['Passenger_Boardings'].sum()

    alighting_totals = chunk.groupby(
        'Station_Name'
    )['Passenger_Alightings'].sum()

    for station, value in boarding_totals.items():
        station_boardings[station] = (
            station_boardings.get(station, 0) + value
        )

    for station, value in alighting_totals.items():
        station_alightings[station] = (
            station_alightings.get(station, 0) + value
        )

station_summary = pd.DataFrame({
    'Total_Boardings': pd.Series(station_boardings),
    'Total_Alightings': pd.Series(station_alightings)
})

station_summary = station_summary.sort_values(
    'Total_Boardings',
    ascending=False
)

station_summary.head(15)

,Total_Boardings,Total_Alightings
Flinders Street,47852560,49229520
Southern Cross,28148000,28823490
Melbourne Central,12381390,11843280
Richmond,10882170,11331100
Parliament,7609980,8687260
Footscray,4989440,4917610
Caulfield,4634580,4709100
Flagstaff,4464500,5249360
North Melbourne,4210170,4506170
South Yarra,3919840,3943570


In [68]:
station_coordinates = {}

for chunk in pd.read_csv(
    filename,
    usecols=[
        'Station_Name',
        'Station_Latitude',
        'Station_Longitude'
    ],
    chunksize=100000
):
    for row in chunk.itertuples(index=False):
        station = row.Station_Name
        lat = row.Station_Latitude
        lon = row.Station_Longitude

        if station not in station_coordinates:
            station_coordinates[station] = {
                'Latitude': set(),
                'Longitude': set()
            }

        if pd.notna(lat):
            station_coordinates[station]['Latitude'].add(lat)

        if pd.notna(lon):
            station_coordinates[station]['Longitude'].add(lon)

coordinate_consistency = pd.DataFrame([
    {
        'Station_Name': station,
        'Unique_Latitudes': len(values['Latitude']),
        'Unique_Longitudes': len(values['Longitude'])
    }
    for station, values in station_coordinates.items()
])

coordinate_consistency[
    (coordinate_consistency['Unique_Latitudes'] > 1) |
    (coordinate_consistency['Unique_Longitudes'] > 1)
]

,Station_Name,Unique_Latitudes,Unique_Longitudes


In [65]:
station_lines = {}

for chunk in pd.read_csv(
    filename,
    usecols=['Station_Name', 'Line_Name'],
    chunksize=100000
):
    for station, group in chunk.groupby('Station_Name'):
        
        if station not in station_lines:
            station_lines[station] = set()
        
        station_lines[station].update(
            group['Line_Name'].dropna().unique()
        )

station_line_summary = pd.Series({
    station: len(lines)
    for station, lines in station_lines.items()
}).sort_values(ascending=False)

print(
    station_line_summary[
        station_line_summary > 1
    ]
)

print(
    "\nStations appearing on more than one line:",
    (station_line_summary > 1).sum()
)

Southern Cross       21
Flinders Street      18
Parliament           14
Melbourne Central    14
Flagstaff            14
                     ..
West Richmond         2
Berwick               2
Canterbury            2
Chatham               2
Yarraman              2
Length: 68, dtype: int64

Stations appearing on more than one line: 68


In [34]:
station_date_pairs = set()

for chunk in pd.read_csv(
    filename,
    usecols=['Station_Name', 'Business_Date'],
    chunksize=100000
):
    pairs = zip(
        chunk['Station_Name'],
        chunk['Business_Date']
    )
    
    station_date_pairs.update(pairs)

station_date_df = pd.DataFrame(
    station_date_pairs,
    columns=['Station_Name', 'Business_Date']
)

station_date_summary = (
    station_date_df
    .groupby('Station_Name')['Business_Date']
    .nunique()
    .sort_values()
)

print(station_date_summary.describe())

count    315.000000
mean     349.320635
std       24.841575
min       39.000000
25%      350.500000
50%      357.000000
75%      360.500000
max      365.000000
Name: Business_Date, dtype: float64


In [35]:
print("Stations with fewer than 300 days:")
print(
    station_date_summary[
        station_date_summary < 300
    ]
)

Stations with fewer than 300 days:
Station_Name
Showgrounds               39
Flemington Racecourse    261
Rosanna                  282
Watsonia                 284
Macleod                  284
Greensborough            284
Montmorency              284
Birregurra               293
Colac                    293
Warrnambool              293
Waurn Ponds              293
Sherwood Park            293
Camperdown               293
South Geelong            293
Winchelsea               293
Marshall                 293
Terang                   293
Name: Business_Date, dtype: int64


In [36]:
import pandas as pd

filename = "train_service_passenger_counts_fy_2024_2025.csv"

station_reference = {}

for chunk in pd.read_csv(
    filename,
    usecols=[
        'Station_Name',
        'Station_Latitude',
        'Station_Longitude',
        'Line_Name'
    ],
    chunksize=100000
):
    
    for station, group in chunk.groupby('Station_Name'):
        
        if station not in station_reference:
            station_reference[station] = {
                'Latitude': group['Station_Latitude'].iloc[0],
                'Longitude': group['Station_Longitude'].iloc[0],
                'Train_Lines': set()
            }
        
        station_reference[station]['Train_Lines'].update(
            group['Line_Name'].dropna().unique()
        )

station_reference_df = pd.DataFrame([
    {
        'Station_Name': station,
        'Latitude': values['Latitude'],
        'Longitude': values['Longitude'],
        'Train_Lines': '; '.join(
            sorted(values['Train_Lines'])
        )
    }
    for station, values in station_reference.items()
])

station_reference_df = station_reference_df.sort_values(
    'Station_Name'
).reset_index(drop=True)

station_reference_df.head(20)

,Station_Name,Latitude,Longitude,Train_Lines
0,Aircraft,-37.866606,144.760809,Werribee
1,Alamein,-37.868320,145.079656,Alamein
2,Albion,-37.777653,144.824704,Sunbury
3,Albury,-36.084262,146.924515,North East
4,Alphington,-37.778394,145.031255,Hurstbridge
5,Altona,-37.867148,144.829645,Werribee
6,Anstey,-37.761242,144.960684,Upfield
7,Ararat,-37.282205,142.936914,Western
8,Ardeer,-37.783063,144.802193,South Western; Western
9,Armadale,-37.856452,145.019326,Cranbourne; Frankston; Pakenham


In [37]:
print("Number of stations:", len(station_reference_df))

Number of stations: 315


In [38]:
station_reference_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315 entries, 0 to 314
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Station_Name  315 non-null    object 
 1   Latitude      315 non-null    float64
 2   Longitude     315 non-null    float64
 3   Train_Lines   315 non-null    object 
dtypes: float64(2), object(2)
memory usage: 10.0+ KB


In [39]:
station_reference_df.isna().sum()

Station_Name    0
Latitude        0
Longitude       0
Train_Lines     0
dtype: int64

In [40]:
station_reference_df.head(20)

,Station_Name,Latitude,Longitude,Train_Lines
0,Aircraft,-37.866606,144.760809,Werribee
1,Alamein,-37.868320,145.079656,Alamein
2,Albion,-37.777653,144.824704,Sunbury
3,Albury,-36.084262,146.924515,North East
4,Alphington,-37.778394,145.031255,Hurstbridge
5,Altona,-37.867148,144.829645,Werribee
6,Anstey,-37.761242,144.960684,Upfield
7,Ararat,-37.282205,142.936914,Western
8,Ardeer,-37.783063,144.802193,South Western; Western
9,Armadale,-37.856452,145.019326,Cranbourne; Frankston; Pakenham


In [41]:
station_reference_df.to_csv(
    "train_station_reference_table.csv",
    index=False
)

print("Station reference table saved successfully.")

Station reference table saved successfully.


In [47]:
stop_locations = pd.read_csv("stop_locations.txt")

print("Rows and columns:", stop_locations.shape)
print("\nColumn names:")
print(stop_locations.columns.tolist())

display(stop_locations.head(10))

Rows and columns: (27613, 1)

Column names:
['867|Weemala Court|Weemala Ct/Plenty River Dr (Greensborough)|Kerbside|Greensborough|3088|Melbourne|Banyule|Greater Metro|-37.689596|145.105088']


,867|Weemala Court|Weemala Ct/Plenty River Dr (Greensborough)|Kerbside|Greensborough|3088|Melbourne|Banyule|Greater Metro|-37.689596|145.105088
0,868|Crana Grove|Crana Gr/Plenty River Dr (Gree...
1,869|Punkerri Circuit|Punkerri Cct/Plenty River...
2,870|Plenty River Drive|231 Plenty River Dr (Gr...
3,875|Oldstead Rd|Oldstead Rd/Diamond Creek Rd (...
4,876|St Thomas PS|St Thomas PS/Diamond Creek Rd...
5,971|Ramu Parade|Ramu Pde/Oriel Rd (Heidelberg ...
6,977|David Myers Building|David Myers Building/...
7,1491|Oriel Road|Oriel Rd/Southern Rd (Heidelbe...
8,1493|Waterdale Road|Waterdale Rd/Southern Rd (...
9,1699|St James Road|St James Rd/Lower Plenty Rd...


In [46]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles in current folder:")
print(os.listdir())

Current folder:
C:\Users\rasab\Downloads\Jupyter

Files in current folder:
['.ipynb_checkpoints', '4.1P.ipynb', 'airlines.csv', 'airports.csv', 'AI_Badges.csv', 'AI_Comments.csv', 'AI_PostHistory.csv', 'AI_PostLinks.csv', 'AI_Posts.csv', 'AI_Tags.csv', 'AI_Users.csv', 'AI_Votes.csv', 'Assignment 1.ipynb', 'BMX_L.xpt', 'BPXO_L.xpt', 'BTC-USD.csv', 'DEMO_L.xpt', 'DIQ_L.xpt', 'DS_Badges.csv', 'DS_Comments.csv', 'DS_PostHistory.csv', 'DS_PostLinks.csv', 'DS_Posts.csv', 'DS_Tags.csv', 'DS_Users.csv', 'DS_Votes.csv', 'EDA.ipynb', 'flights.csv', 'flights.db', 'microclimate-sensors-data.csv', 'myki_2017_top20_tram_routes_ScanOffTransaction.csv', 'myki_2017_top20_tram_routes_ScanOnTransaction.csv', 'ObesityDataSet_raw_and_data_sinthetic.csv', 'outputs', 'planes.csv', 'P_DIQ.xpt', 'P_SMQ.xpt', 'sample_ecg.csv', 'SIT742-2026T2-A1-S225287928.ipynb', 'SMQ_L.xpt', 'stop_locations.txt', 'Task - 2P.ipynb', 'Task 2.1p.ipynb', 'Task 2P new.ipynb', 'Task 3P.ipynb', 'Task 7D.ipynb', 'Task 8HD.ipynb', 'Tas

In [48]:
stop_locations = pd.read_csv(
    "stop_locations.txt",
    sep="|",
    header=None
)

print(stop_locations.shape)
display(stop_locations.head())

(27614, 11)


,0,1,2,3,4,5,6,7,8,9,10
0,867,Weemala Court,Weemala Ct/Plenty River Dr (Greensborough),Kerbside,Greensborough,3088.0,Melbourne,Banyule,Greater Metro,-37.689596,145.105088
1,868,Crana Grove,Crana Gr/Plenty River Dr (Greensborough),Kerbside,Greensborough,3088.0,Melbourne,Banyule,Greater Metro,-37.686742,145.105588
2,869,Punkerri Circuit,Punkerri Cct/Plenty River Dr (Greensborough),Kerbside,Greensborough,3088.0,Melbourne,Banyule,Greater Metro,-37.683643,145.108743
3,870,Plenty River Drive,231 Plenty River Dr (Greensborough),Kerbside,Greensborough,3088.0,Melbourne,Banyule,Greater Metro,-37.682591,145.111331
4,875,Oldstead Rd,Oldstead Rd/Diamond Creek Rd (Greensborough),Kerbside,Greensborough,3088.0,Melbourne,Banyule,Greater Metro,-37.685336,145.117319


In [49]:
print(stop_locations.iloc[:, 0].head(20).tolist())

[867, 868, 869, 870, 875, 876, 971, 977, 1491, 1493, 1699, 1700, 1701, 1702, 1703, 1704, 1705, 1706, 1707, 1708]


In [50]:
# Get the station names from your existing reference table
rail_stations = set(
    station_reference_df['Station_Name']
    .str.strip()
    .str.lower()
)

# Find stop records whose stop name matches a railway station name
matching_stops = stop_locations[
    stop_locations[1]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(rail_stations)
].copy()

print("Matching records:", len(matching_stops))

display(matching_stops.head(20))

Matching records: 325


,0,1,2,3,4,5,6,7,8,9,10
391,19545,Melbourne Central,Melbourne Central/Lonsdale St (Melbourne City),Kerbside,Melbourne City,3000.0,Melbourne,Melbourne,Greater Metro,-37.811564,144.965029
643,20302,Chiltern,Chiltern Railway Station (Chiltern),Platform,Chiltern,3683.0,Indigo,Indigo,Goulburn & Ovens-Murray,-36.155634,146.611380
929,22253,Essendon,Essendon Railway Station (Essendon),Platform,Essendon,3040.0,Melbourne,Moonee Valley,Greater Metro,-37.756010,144.916200
933,20288,Ararat,Ararat Railway Station (Ararat),Platform,Ararat,3377.0,Ararat,Ararat,Central Highlands & Wimmera,-37.282202,142.936912
934,20300,Camperdown,Camperdown Railway Station (Camperdown),Platform,Camperdown,3260.0,Corangamite,Corangamite,Barwon & Western District,-38.228900,143.150932
935,20318,Kerang,Kerang Railway Station (Kerang),Platform,Kerang,3579.0,Gannawarra,Gannawarra,Loddon & Mallee,-35.733123,143.924430
936,20349,Terang,Terang Railway Station (Terang),Platform,Terang,3264.0,Corangamite,Corangamite,Barwon & Western District,-38.236215,142.911467
1070,19948,Sandringham,Sandringham Railway Station (Sandringham),Platform,Sandringham,3191.0,Melbourne,Bayside,Greater Metro,-37.950328,145.004566
1071,19949,Hampton,Hampton Railway Station (Hampton),Platform,Hampton,3188.0,Melbourne,Bayside,Greater Metro,-37.937973,145.001466
1072,19950,Brighton Beach,Brighton Beach Railway Station (Brighton),Platform,Brighton,3186.0,Melbourne,Bayside,Greater Metro,-37.926482,144.989157


In [51]:
print("Matching records:", len(matching_stops))

display(
    matching_stops[
        [0, 1, 2, 3, 9, 10]
    ].head(30)
)

print(
    "Unique matching stop names:",
    matching_stops[1].nunique()
)

Matching records: 325


,0,1,2,3,9,10
391,19545,Melbourne Central,Melbourne Central/Lonsdale St (Melbourne City),Kerbside,-37.811564,144.965029
643,20302,Chiltern,Chiltern Railway Station (Chiltern),Platform,-36.155634,146.611380
929,22253,Essendon,Essendon Railway Station (Essendon),Platform,-37.756010,144.916200
933,20288,Ararat,Ararat Railway Station (Ararat),Platform,-37.282202,142.936912
934,20300,Camperdown,Camperdown Railway Station (Camperdown),Platform,-38.228900,143.150932
935,20318,Kerang,Kerang Railway Station (Kerang),Platform,-35.733123,143.924430
936,20349,Terang,Terang Railway Station (Terang),Platform,-38.236215,142.911467
1070,19948,Sandringham,Sandringham Railway Station (Sandringham),Platform,-37.950328,145.004566
1071,19949,Hampton,Hampton Railway Station (Hampton),Platform,-37.937973,145.001466
1072,19950,Brighton Beach,Brighton Beach Railway Station (Brighton),Platform,-37.926482,144.989157


Unique matching stop names: 291


In [52]:
# Keep only records classified as railway platforms
railway_stops = stop_locations[
    stop_locations[3].astype(str).str.strip().str.lower() == "platform"
].copy()

print("Total railway platform records:", len(railway_stops))

# Match railway platforms against our 315 station reference names
railway_station_names = set(
    station_reference_df['Station_Name']
    .str.strip()
    .str.lower()
)

railway_matches = railway_stops[
    railway_stops[1]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(railway_station_names)
].copy()

print("Matching railway platform records:", len(railway_matches))
print(
    "Unique matching railway station names:",
    railway_matches[1].nunique()
)

display(
    railway_matches[
        [0, 1, 2, 3, 9, 10]
    ].head(30)
)

Total railway platform records: 862
Matching railway platform records: 305
Unique matching railway station names: 288


,0,1,2,3,9,10
643,20302,Chiltern,Chiltern Railway Station (Chiltern),Platform,-36.155634,146.611380
929,22253,Essendon,Essendon Railway Station (Essendon),Platform,-37.756010,144.916200
933,20288,Ararat,Ararat Railway Station (Ararat),Platform,-37.282202,142.936912
934,20300,Camperdown,Camperdown Railway Station (Camperdown),Platform,-38.228900,143.150932
935,20318,Kerang,Kerang Railway Station (Kerang),Platform,-35.733123,143.924430
936,20349,Terang,Terang Railway Station (Terang),Platform,-38.236215,142.911467
1070,19948,Sandringham,Sandringham Railway Station (Sandringham),Platform,-37.950328,145.004566
1071,19949,Hampton,Hampton Railway Station (Hampton),Platform,-37.937973,145.001466
1072,19950,Brighton Beach,Brighton Beach Railway Station (Brighton),Platform,-37.926482,144.989157
1073,19951,Middle Brighton,Middle Brighton Railway Station (Brighton),Platform,-37.915135,144.996298


In [53]:
platform_counts = (
    railway_matches
    .groupby(1)[0]
    .nunique()
    .sort_values(ascending=False)
)

print("Stations with multiple platform IDs:")
display(
    platform_counts[
        platform_counts > 1
    ].head(30)
)

Stations with multiple platform IDs:


1
Footscray           2
Pakenham            2
Broadmeadows        2
Sunbury             2
Ginifer             2
Caulfield           2
St Albans           2
Berwick             2
Springhurst         2
Clayton             2
Southern Cross      2
Watergardens        2
Flinders Street     2
North Melbourne     2
Richmond            2
Dandenong           2
Essendon            2
Name: 0, dtype: int64

In [54]:
# Create one station-level record with all available platform IDs

railway_station_reference = (
    railway_matches
    .groupby(1)
    .agg({
        0: lambda x: '; '.join(
            sorted(x.astype(str).unique())
        ),
        9: 'mean',
        10: 'mean'
    })
    .reset_index()
)

railway_station_reference.columns = [
    'Station_Name',
    'Station_IDs',
    'GTFS_Latitude',
    'GTFS_Longitude'
]

# Add the train-line information from our existing 315-station table
final_station_reference = station_reference_df.merge(
    railway_station_reference,
    on='Station_Name',
    how='left'
)

final_station_reference.head(20)

,Station_Name,Latitude,Longitude,Train_Lines,Station_IDs,GTFS_Latitude,GTFS_Longitude
0,Aircraft,-37.866606,144.760809,Werribee,NaN,NaN,NaN
1,Alamein,-37.868320,145.079656,Alamein,19847,-37.868317,145.079659
2,Albion,-37.777653,144.824704,Sunbury,20004,-37.777655,144.824709
3,Albury,-36.084262,146.924515,North East,NaN,NaN,NaN
4,Alphington,-37.778394,145.031255,Hurstbridge,19931,-37.778396,145.031251
5,Altona,-37.867148,144.829645,Werribee,19926,-37.867151,144.829647
6,Anstey,-37.761242,144.960684,Upfield,19967,-37.761239,144.960686
7,Ararat,-37.282205,142.936914,Western,20288,-37.282202,142.936912
8,Ardeer,-37.783063,144.802193,South Western; Western,20020,-37.783063,144.802193
9,Armadale,-37.856452,145.019326,Cranbourne; Frankston; Pakenham,19945,-37.856452,145.019328


In [55]:
print(
    "Total stations:",
    len(final_station_reference)
)

print(
    "Stations with railway IDs:",
    final_station_reference['Station_IDs'].notna().sum()
)

print(
    "Stations without railway IDs:",
    final_station_reference['Station_IDs'].isna().sum()
)

Total stations: 315
Stations with railway IDs: 274
Stations without railway IDs: 41


In [56]:
unmatched_stations = final_station_reference[
    final_station_reference['Station_IDs'].isna()
]

print(
    "Unmatched stations:",
    len(unmatched_stations)
)

display(
    unmatched_stations[
        [
            'Station_Name',
            'Latitude',
            'Longitude',
            'Train_Lines'
        ]
    ]
)

Unmatched stations: 41


,Station_Name,Latitude,Longitude,Train_Lines
0,Aircraft,-37.866606,144.760809,Werribee
3,Albury,-36.084262,146.924515,North East
28,Bendigo,-36.765673,144.283012,Northern
40,Brunswick,-37.767721,144.959586,Upfield
49,Caroline Springs,-37.767153,144.738023,Western
61,Cobblebank,-37.712542,144.604103,Western
65,Coolaroo,-37.661003,144.926056,Craigieburn
77,Dennis,-37.779187,145.008242,Hurstbridge
87,East Pakenham,-38.084284,145.506311,Eastern; Pakenham
95,Epsom,-36.706343,144.321036,Northern


In [57]:
final_station_reference['ID_Match_Status'] = final_station_reference[
    'Station_IDs'
].apply(
    lambda x: 'Matched' if pd.notna(x) else 'Unmatched'
)

final_station_reference['ID_Match_Status'].value_counts()

ID_Match_Status
Matched      274
Unmatched     41
Name: count, dtype: int64

In [58]:
final_station_reference.to_csv(
    "train_station_reference_for_matching.csv",
    index=False
)

print("Final station reference table updated and saved.")

Final station reference table updated and saved.


In [59]:
import numpy as np

final_station_reference['Latitude_Difference'] = (
    final_station_reference['Latitude'] -
    final_station_reference['GTFS_Latitude']
).abs()

final_station_reference['Longitude_Difference'] = (
    final_station_reference['Longitude'] -
    final_station_reference['GTFS_Longitude']
).abs()

display(
    final_station_reference[
        ['Station_Name',
         'Latitude_Difference',
         'Longitude_Difference']
    ].describe()
)

,Latitude_Difference,Longitude_Difference
count,2.740000e+02,2.740000e+02
mean,5.490876e-06,1.744526e-05
std,2.504002e-05,1.293608e-04
min,0.000000e+00,0.000000e+00
25%,1.000000e-06,1.000000e-06
50%,2.000000e-06,2.000000e-06
75%,3.000000e-06,4.000000e-06
max,2.410000e-04,1.689000e-03


In [60]:
print(
    "Stations with reference coordinates:",
    final_station_reference['GTFS_Latitude'].notna().sum()
)

print(
    "Maximum latitude difference:",
    final_station_reference['Latitude_Difference'].max()
)

print(
    "Maximum longitude difference:",
    final_station_reference['Longitude_Difference'].max()
)

Stations with reference coordinates: 274
Maximum latitude difference: 0.00024099999999549482
Maximum longitude difference: 0.0016889999999989413


In [61]:
final_station_reference.to_csv(
    "train_station_reference_for_matching.csv",
    index=False
)

print("Final station reference table saved successfully.")

Final station reference table saved successfully.
